# Curriculum LC->NX edit model (route+conn+adj, norm-Huber critic)

Edit-агент учится по **curriculum** от «много чинить» к «мало/ничего»:
**nx (≈61% избыточности) → lc_mid (K=4) → lc_low (K=2) → lc_clean (K=0)**.
Сначала агент осваивает редактирование на сильно избыточных nx-сетях (без halt-коллапса),
затем постепенно добавляются всё более «чистые» LC-сети (нужна сдержанность/точечная чистка).

Активная стадия логируется (`curriculum_stage`) — на кривых обучения видно, в какой момент
сменилась задача и как поехал reward. Objective: **route+conn+adj** (demand off), adj фикс
(cap, target=0.15, W=2), cost-веса route/conn варьируются, критик **norm+Huber+value-clip**.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"   # выбрать GPU 1
import sys, pickle, shutil, random as _random
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from hydra import compose, initialize_config_dir
from IPython.display import display

from eval_lib.context import (ROOT_DIR, CFG_DIR, DATASETS_DIR,
                              MODEL_OUTPUTS_DIR, EDIT_MODEL_WEIGHTS_DIR)
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))
from connectpt.routes_generator import utils as lrnu
from connectpt.routes_generator.improvement_learning import (
    load_raw_graphs_and_lc_routes, make_improvement_batch,
    rollout_lc_improvement, train_lc_improvement_cfg)
from connectpt.routes_generator.torch_utils import (
    get_batch_tensor_from_routes, dump_routes)
from connectpt.routes_generator.transit_time_estimator import RouteGenBatchState
from connectpt.routes_generator.citygraph_dataset import (
    STOP_KEY, DynamicCityGraphDataset)
from connectpt.routes_generator.nx_heuristic import build_nx_heuristic_routes
from connectpt.routes_generator.bee_colony import get_adjustment_degrees
from torch_geometric.data import Batch
from eval_lib.results_io import save_table
from eval_lib import build_lc_cfg, run_lc, as_route_tensor

pd.set_option("display.max_columns", None)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## Конфигурация

In [ ]:
# --- датасет: 4 tier'а по сложности (easy->hard), по 250 графов ---
N_GRAPHS        = 1000
RAW_N_NODES     = 50
RAW_GRAPH_TYPE  = "mixed"
RAW_GRAPH_SEED  = 0
TARGET_N_ROUTES = 12
MIN_ROUTE_LEN   = 8
MAX_ROUTE_LEN   = 15
LC_N_SAMPLES    = 1
LC_COMBOS = [(1.0,0.0,0.0,"demand"), (0.0,1.0,0.0,"route"), (0.0,0.0,1.0,"conn")]
TIERS   = ["nx", "lc_mid", "lc_low", "lc_clean"]   # 250 каждый
TIER_K  = {"lc_mid": 4, "lc_low": 2, "lc_clean": 0}  # K инъекций (nx -- свой генератор)
DATASET_DIRNAME = "lc_curriculum_n50_r12_len8_15"
NEW_DATASET_DIR = DATASETS_DIR / DATASET_DIRNAME
SUBSET_PKL = NEW_DATASET_DIR / "raw_graphs_subset.pkl"
META_CSV   = NEW_DATASET_DIR / "meta.csv"
FORCE_REGEN = False

# --- objective route+conn+adj ---
DISABLED_COST_COMPONENTS = ["demand"]
ADJ_OBJECTIVE = "cap"; ADJ_WEIGHT = 2.0; ADJ_TARGET = 0.15; ADJ_MODE = "paper"; ADJ_GAP = 0.1
VARY_WEIGHTS = True; OP_FRACTION = 0.4; MCW_FRACTION = 0.4   # route/conn веса варьируются

# --- анти-halt-collapse ---
FORCE_NONHALT_FIRST_STEP = True
FORCE_NONHALT_UNTIL_ITER = 50      # форсировать правки первые 50 эпох (стадия nx)
ENTROPY_WEIGHT = 0.01

# --- критик norm+Huber+value-clip ---
CRITIC_OVERRIDES = ["++critic_normalize_returns=true", "++critic_huber=true",
                    "++critic_huber_delta=1.0", "++critic_value_clip=0.2"]

# --- обучение ---
N_ITERATIONS = 300
BATCH_SIZE   = 16
TRAIN_FRACTION = 0.9
SPLIT_SEED   = 0
MAX_ROUTE_EDIT_STEPS = MAX_ROUTE_LEN
MAX_TRIM_ACTIONS_PER_ROUTE = 1

# --- curriculum: аккумулятивно добавляем более сложные tier'ы ---
USE_CURRICULUM = True
CURRICULUM = [
    (50,           ["nx"],                              "nx"),
    (100,          ["nx", "lc_mid"],                    "nx+mid"),
    (150,          ["nx", "lc_mid", "lc_low"],          "+low"),
    (N_ITERATIONS, ["nx", "lc_mid", "lc_low", "lc_clean"], "all"),
]

# --- eval ---
EVAL_WEIGHT_COMBOS = [(1.0, 0.0, "route-only"), (0.0, 1.0, "conn-only"), (0.5, 0.5, "balanced")]
EVAL_N_PER_TIER = 10
RUN_NAME = "lc_curriculum_route_conn_adj"
print(f"{N_GRAPHS} graphs, tiers={TIERS}; curriculum={[c[2]+'<'+str(c[0]) for c in CURRICULUM]}")
print(f"adj cap t={ADJ_TARGET} W={ADJ_WEIGHT}; force_nonhalt<{FORCE_NONHALT_UNTIL_ITER}; entropy={ENTROPY_WEIGHT}")

## Генерация датасета (4 tier'а на диск)

- **nx**: `build_nx_heuristic_routes` (сильно избыточные maxlen-маршруты);
- **lc_mid/low/clean**: LC (крайний weight-combo) + `K` инъекций (over-extend / out-and-back).

Сохраняется в формате `load_raw_graphs_and_lc_routes` + `meta.csv` (tier, lc_combo, K, redund%).

In [ ]:
def _to_fixed(routes):
    t = as_route_tensor(routes).long()
    if t.ndim == 3:
        t = t[0]
    if t.shape[0] < TARGET_N_ROUTES:
        t = torch.cat([t, torch.full((TARGET_N_ROUTES - t.shape[0], t.shape[1]), -1, dtype=t.dtype)], 0)
    else:
        t = t[:TARGET_N_ROUTES]
    if t.shape[1] < MAX_ROUTE_LEN:
        t = torch.cat([t, torch.full((t.shape[0], MAX_ROUTE_LEN - t.shape[1]), -1, dtype=t.dtype)], 1)
    elif t.shape[1] > MAX_ROUTE_LEN:
        t = t[:, :MAX_ROUTE_LEN]
    return t


def _tensors(g):
    return {"node_locs": g[STOP_KEY].pos.detach().cpu().clone(),
            "street_adj": g.street_adj.detach().cpu().clone(),
            "demand": g.demand.detach().cpu().clone()}


def _redundancy(routes):
    cnt = Counter()
    for r in routes.tolist():
        r = [x for x in r if x >= 0]
        for a, b in zip(r[:-1], r[1:]):
            cnt[(min(a, b), max(a, b))] += 1
    tot = sum(cnt.values())
    return 0.0 if tot == 0 else (tot - len(cnt)) / tot


def inject_redundancy(routes, street_adj, n_inject, rng):
    adj = torch.isfinite(street_adj) & (street_adj > 0)
    routes = routes.clone()
    for _ in range(n_inject):
        lens = (routes >= 0).sum(1)
        cand = [r for r in range(routes.shape[0]) if 2 <= int(lens[r]) <= MAX_ROUTE_LEN - 2]
        if not cand:
            break
        r = rng.choice(cand); l = int(lens[r]); last = int(routes[r, l - 1])
        nbrs = [x for x in torch.where(adj[last])[0].tolist() if x != last]
        if not nbrs:
            continue
        y = rng.choice(nbrs)
        routes[r, l] = y; routes[r, l + 1] = last   # out-and-back (дубль ребра)
    return routes


def _nx_routes(g, seed):
    try:
        return build_nx_heuristic_routes(g, num_routes=TARGET_N_ROUTES,
                                         min_len=MIN_ROUTE_LEN, max_len=MAX_ROUTE_LEN, seed=seed)
    except ValueError:
        return build_nx_heuristic_routes(g, num_routes=TARGET_N_ROUTES,
                                         min_len=2, max_len=MAX_ROUTE_LEN, seed=seed)


def generate_dataset():
    if NEW_DATASET_DIR.exists():
        shutil.rmtree(NEW_DATASET_DIR)
    NEW_DATASET_DIR.mkdir(parents=True, exist_ok=True)
    _random.seed(RAW_GRAPH_SEED); torch.manual_seed(RAW_GRAPH_SEED)
    ds = DynamicCityGraphDataset(min_nodes=RAW_N_NODES, max_nodes=RAW_N_NODES,
                                 data_type=RAW_GRAPH_TYPE, mumford_style=True, pos_only=False)
    raw = [ds.generate_graph(n_nodes=RAW_N_NODES) for _ in range(N_GRAPHS)]
    per = N_GRAPHS // len(TIERS)
    subset, meta = [], []
    for gi, g in enumerate(raw):
        tier = TIERS[min(gi // per, len(TIERS) - 1)]
        rng = _random.Random(1000 + gi)
        if tier == "nx":
            routes = _to_fixed(_nx_routes(g, RAW_GRAPH_SEED + gi)); ctag = "nx"; K = 0
            rb = _redundancy(routes); ra = rb
        else:
            d, rt, cn, ctag = LC_COMBOS[gi % len(LC_COMBOS)]
            c = build_lc_cfg(run_name=f"cur_{gi}", n_routes=TARGET_N_ROUTES,
                             min_route_len=MIN_ROUTE_LEN, max_route_len=MAX_ROUTE_LEN,
                             demand_time_weight=d, route_time_weight=rt, median_connectivity_weight=cn)
            _, _, _, r, _ = run_lc(c, tensors=_tensors(g), run_name_prefix="cur_", n_samples=LC_N_SAMPLES)
            routes = _to_fixed(r); rb = _redundancy(routes)
            K = TIER_K[tier]; routes = inject_redundancy(routes, g.street_adj, K, rng); ra = _redundancy(routes)
        gdir = NEW_DATASET_DIR / f"graph_{gi:04d}"; gdir.mkdir(parents=True, exist_ok=True)
        dump_routes(f"lc_cur_graph_{gi:04d}_routes", routes, out_dir=gdir)
        subset.append(g)
        meta.append({"graph_index": gi, "tier": tier, "lc_combo": ctag, "K": K,
                     "redun_before": round(rb, 4), "redun_after": round(ra, 4)})
        if (gi + 1) % 100 == 0:
            print(f"  {gi+1}/{N_GRAPHS} (tier={tier})")
    with SUBSET_PKL.open("wb") as fh:
        pickle.dump(subset, fh)
    pd.DataFrame(meta).to_csv(META_CSV, index=False)
    print(f"Saved {len(subset)} graphs -> {NEW_DATASET_DIR}")


_have = len(list(NEW_DATASET_DIR.glob("graph_*"))) if NEW_DATASET_DIR.exists() else 0
if SUBSET_PKL.exists() and _have == N_GRAPHS and META_CSV.exists() and not FORCE_REGEN:
    print(f"Датасет уже на диске ({_have}) -> пропуск")
else:
    if _have and _have != N_GRAPHS:
        print(f"На диске {_have}, нужно {N_GRAPHS} -> перегенерация")
    generate_dataset()

## Чтение + split + curriculum_fn

In [ ]:
graphs, seed_routes = load_raw_graphs_and_lc_routes(SUBSET_PKL, NEW_DATASET_DIR)
meta_df = pd.read_csv(META_CSV)
N = len(graphs)
print(f"loaded {N} graphs; seed_routes={tuple(seed_routes.shape)}")
print("redundancy after injection by tier:")
display(meta_df.groupby("tier")[["K", "redun_before", "redun_after"]].mean().round(3)
        .reindex(TIERS))

_perm = torch.randperm(N, generator=torch.Generator().manual_seed(SPLIT_SEED))
_ntr = int(TRAIN_FRACTION * N)
TRAIN_INDICES = _perm[:_ntr].clone()
VAL_INDICES = _perm[_ntr:].clone()
TIER_OF = dict(zip(meta_df["graph_index"], meta_df["tier"]))

# индексы train по tier'ам
_train_by_tier = {t: [] for t in TIERS}
for gi in TRAIN_INDICES.tolist():
    _train_by_tier[TIER_OF[gi]].append(gi)
_train_by_tier = {t: torch.tensor(v, dtype=torch.long) for t, v in _train_by_tier.items()}
print("train graphs per tier:", {t: len(v) for t, v in _train_by_tier.items()})


def curriculum_fn(iteration):
    """iteration -> (active train indices, stage_label) по расписанию CURRICULUM."""
    for until, tiers, label in CURRICULUM:
        if iteration < until:
            idx = torch.cat([_train_by_tier[t] for t in tiers if len(_train_by_tier[t])])
            return idx, label
    tiers = CURRICULUM[-1][1]
    idx = torch.cat([_train_by_tier[t] for t in tiers if len(_train_by_tier[t])])
    return idx, CURRICULUM[-1][2]


def stage_spans():
    """[(start_iter, end_iter, label)] для заливки на кривых (1-индексация эпох)."""
    spans, prev = [], 0
    for until, _t, label in CURRICULUM:
        spans.append((prev + 1, until, label)); prev = until
    return spans

## Модель + cost (route+conn+adj, варьируемые веса, norm-Huber критик)

In [ ]:
overrides = [
    "model=bestsofar_feb2023_trim",
    "model.route_generator.kwargs.serial_halting=True",
    f"++run_name={RUN_NAME}", "++experiment.logdir=null",
    f"++adjustment_degree_weight={float(ADJ_WEIGHT)}",
    f"++adjustment_degree_target={float(ADJ_TARGET)}",
    f"++adjustment_degree_objective={ADJ_OBJECTIVE}",
    f"++adjustment_degree_gap={float(ADJ_GAP)}",
    f"++adjustment_degree_mode={ADJ_MODE}",
    f"++entropy_weight={float(ENTROPY_WEIGHT)}",
    f"++force_nonhalt_first_step_until_iter={int(FORCE_NONHALT_UNTIL_ITER)}",
] + CRITIC_OVERRIDES
with initialize_config_dir(config_dir=str(CFG_DIR), version_base=None):
    cfg = compose(config_name="ppo_50nodes.yaml", overrides=overrides)
device, run_name, _, cost_obj, model = lrnu.process_standard_experiment_cfg(
    cfg, run_name_prefix="improvement_")
cost_obj.ignore_stops_oob = True
cost_obj.set_enabled_components(disabled_components=DISABLED_COST_COMPONENTS or None)
if VARY_WEIGHTS:
    cost_obj.variable_weights = True
    cost_obj.pp_fraction = 0.0; cost_obj.op_fraction = OP_FRACTION; cost_obj.mcw_fraction = MCW_FRACTION
BEST_MODEL_PATH = EDIT_MODEL_WEIGHTS_DIR / f"{run_name}.pt"
print(f"run_name={run_name} | enabled={list(cost_obj.enabled_component_names)} | "
      f"variable_weights={cost_obj.variable_weights}")
print(f"critic norm={cfg.get('critic_normalize_returns')} huber={cfg.get('critic_huber')} clip={cfg.get('critic_value_clip')}")

## Обучение (300 эпох, curriculum)

In [ ]:
train_result = train_lc_improvement_cfg(
    model=model, cost_obj=cost_obj, graphs=graphs, seed_routes=seed_routes,
    device=device, cfg=cfg, output_dir=MODEL_OUTPUTS_DIR, run_name=run_name,
    train_fraction=TRAIN_FRACTION, batch_size=BATCH_SIZE,
    min_route_len=MIN_ROUTE_LEN, max_route_len=MAX_ROUTE_LEN, seed=SPLIT_SEED,
    max_route_edit_steps=MAX_ROUTE_EDIT_STEPS,
    max_trim_actions_per_route=MAX_TRIM_ACTIONS_PER_ROUTE,
    target_n_routes=TARGET_N_ROUTES,
    train_indices=TRAIN_INDICES, val_indices=VAL_INDICES,
    best_model_path=BEST_MODEL_PATH, n_iterations=N_ITERATIONS,
    force_nonhalt_first_step=FORCE_NONHALT_FIRST_STEP,
    curriculum_fn=(curriculum_fn if USE_CURRICULUM else None),
)
history_df = pd.DataFrame(train_result["history"])
save_table(history_df, f"{RUN_NAME}_training_history")
print(f"history rows={len(history_df)}; best -> {BEST_MODEL_PATH}")

## Кривые актора (с границами стадий curriculum)

In [ ]:
h = history_df
def _num(col):
    return pd.to_numeric(h[col], errors="coerce") if col in h.columns else None

_spans = stage_spans()
_colors = ["#eaf3ff", "#eafbea", "#fff6e6", "#fdeaea", "#f0eaff"]
def _shade(ax):
    for k, (s, e, lab) in enumerate(_spans):
        ax.axvspan(s, e, color=_colors[k % len(_colors)], alpha=0.6, zorder=0)
        ax.axvline(s, color="gray", lw=0.6, ls=":")

fig, ax = plt.subplots(2, 3, figsize=(17, 8), constrained_layout=True)
panels = [("train_reward_mean","train reward"), ("val_delta","val cost delta (+=улучш.)"),
          ("val_win_rate","val win rate"), ("train_action_avg_actions_per_route","avg edits/route"),
          ("val_component_delta_route","val route delta"),
          ("val_component_delta_connectivity","val conn delta")]
for a,(col,title) in zip(ax.flat, panels):
    _shade(a); y=_num(col)
    if y is not None and y.notna().any():
        a.plot(h["epoch"], y, marker="o", ms=2, color="tab:blue", zorder=3)
    a.axhline(0,color="k",lw=0.7); a.set_title(title); a.set_xlabel("epoch"); a.grid(alpha=0.2)
# подписи стадий сверху
for s,e,lab in _spans:
    ax[0,0].text((s+e)/2, ax[0,0].get_ylim()[1], lab, ha="center", va="bottom", fontsize=8)
fig.suptitle("Actor curves + curriculum stages (заливка = стадия)", fontsize=13, fontweight="bold")
plt.show(); plt.close(fig)

## Метрики критика (с границами стадий)

In [ ]:
crit_cols = [c for c in h.columns if "critic" in c.lower()]
print("critic columns:", crit_cols)
if crit_cols:
    n=len(crit_cols)
    fig, ax = plt.subplots(1, n, figsize=(5*n, 4), squeeze=False, constrained_layout=True)
    for a, col in zip(ax[0], crit_cols):
        _shade(a); y=_num(col)
        if y is not None and y.notna().any():
            a.plot(h["epoch"], y, marker="o", ms=2, color="tab:orange", zorder=3)
        a.set_title(col, fontsize=9); a.set_xlabel("epoch"); a.grid(alpha=0.2)
        if "explained" in col: a.axhline(0, color="k", lw=0.7)
    fig.suptitle("Critic metrics + curriculum stages", fontsize=13, fontweight="bold")
    plt.show(); plt.close(fig)
    display(h[["epoch","curriculum_stage"]+crit_cols].iloc[::max(1,len(h)//15)].round(4))

## Оценка по tier'ам + redundancy до/после + граничные веса

Финальная модель на val по каждому tier'у и граничным cost-весам. Главное —
**redun_drop>0** (агент убирает избыточность), особенно на nx/lc_mid; и насколько
он бережёт чистый lc_clean (малый adj).

In [ ]:
def _redun_t(routes_2d):
    cnt = Counter()
    for r in routes_2d.tolist():
        r = [x for x in r if x >= 0]
        for a, b in zip(r[:-1], r[1:]):
            cnt[(min(a, b), max(a, b))] += 1
    tot = sum(cnt.values())
    return 0.0 if tot == 0 else (tot - len(cnt)) / tot

val_by_tier = {t: [] for t in TIERS}
for gi in VAL_INDICES.tolist():
    t = TIER_OF[gi]
    if len(val_by_tier[t]) < EVAL_N_PER_TIER:
        val_by_tier[t].append(gi)

base_w = cost_obj.get_weights(device)
def mkw(rw, cw):
    w = {k: (v.clone() if torch.is_tensor(v) else v) for k, v in base_w.items()}
    w["demand_time_weight"] = torch.as_tensor(0.0, device=device)
    w["route_time_weight"] = torch.as_tensor(float(rw), device=device)
    w["median_connectivity_weight"] = torch.as_tensor(float(cw), device=device)
    return w

rows = []
model.eval()
for rw, cw, wtag in EVAL_WEIGHT_COMBOS:
    weights = mkw(rw, cw)
    for t in TIERS:
        idxs = val_by_tier[t]
        if not idxs:
            continue
        rb_b, rb_a, adjs = [], [], []
        for gi in idxs:
            gb, rb = make_improvement_batch(graphs, seed_routes, torch.tensor([gi]), device,
                                            training=False, target_n_routes=TARGET_N_ROUTES)
            with torch.no_grad():
                out = rollout_lc_improvement(model, cost_obj, gb, rb, MIN_ROUTE_LEN, MAX_ROUTE_LEN,
                                             greedy=True, cost_weights=weights,
                                             max_route_edit_steps=MAX_ROUTE_EDIT_STEPS,
                                             max_trim_actions_per_route=MAX_TRIM_ACTIONS_PER_ROUTE)
            imp = get_batch_tensor_from_routes(out[0].routes, device, max_route_len=rb.shape[-1])
            rb_b.append(_redun_t(rb[0])); rb_a.append(_redun_t(imp[0]))
            nr = min(imp.shape[1], rb.shape[1]); ll = min(imp.shape[-1], rb.shape[-1])
            adjs.append(get_adjustment_degrees(imp[:, :nr, :ll], rb[:, :nr, :ll],
                        cost_obj.symmetric_routes, gap=ADJ_GAP, mode=ADJ_MODE).mean().item())
        rows.append({"eval_weights": wtag, "tier": t, "n": len(idxs),
                     "redun_seed": round(float(np.mean(rb_b)), 3),
                     "redun_after": round(float(np.mean(rb_a)), 3),
                     "redun_drop": round(float(np.mean(rb_b) - np.mean(rb_a)), 3),
                     "adj_vs_seed": round(float(np.mean(adjs)), 3)})
eval_df = pd.DataFrame(rows)
display(eval_df)
save_table(eval_df, f"{RUN_NAME}_eval_by_tier")
print("redun_drop>0 => агент убирает избыточность; смотри nx/lc_mid (много дублей) vs lc_clean.")